# A4 — đo "app lạ" (HUONG_DAN_GPU_08-11, bước 2)
**Trước khi chạy cell đầu tiên:** đặt notebook **Private** (Settings), Internet **Off**, Accelerator **GPU (cùng loại RTX PRO 6000 như các lần trước)**.
Input cần gắn: `lifemate-code` (**bản mới**, từ commit 17b090a), raw (`train+public_test`), 2 model dataset (Qwen3-4B, PhoWhisper), và output notebook **Hello_done**.
Chạy tay P-1 đến A4-2 trước (khoảng 15 phút), **gửi dòng torch ở P-2 cho Phương ngay**, rồi dừng phiên và dùng *Save & Run All (Commit)*.
Chỉ so các cấu hình A4 với nhau, **không** so với điểm dev 56,98 của `q4b_lora` (A4 dùng ASR `phowl`, `q4b_lora` đánh giá với `phowl_ft`).

In [ ]:
%%bash
# P-1: setup. Cần gắn: lifemate-code (BẢN MỚI), raw, 2 model dataset, và output notebook Hello_done (state/ và checkpoints/)
SRC=$(find /kaggle/input -name kaggle_setup.sh -print -quit)
[ -n "$SRC" ] || { echo "ERROR: không thấy kaggle_setup.sh, dataset code đã gắn chưa?"; exit 1; }
QW=$(grep -rlE --include=config.json '"model_type"[[:space:]]*:[[:space:]]*"qwen3"' /kaggle/input 2>/dev/null | head -1 | xargs -r dirname)
PH=$(grep -rlE --include=config.json '"model_type"[[:space:]]*:[[:space:]]*"whisper"' /kaggle/input 2>/dev/null | head -1 | xargs -r dirname)
echo "Qwen3 dir  : ${QW:-KHÔNG THẤY}"; echo "Whisper dir: ${PH:-KHÔNG THẤY}"
[ -n "$QW" ] && [ -n "$PH" ] || { echo "ERROR: thiếu thư mục model"; exit 1; }
cp "$SRC" /kaggle/working/setup_patched.sh
sed -i -e "s#^QWEN=.*#QWEN=$QW#" -e "s#^PHO=.*#PHO=$PH#" -e 's/python -I /python /g' /kaggle/working/setup_patched.sh
bash /kaggle/working/setup_patched.sh
sed -i 's/python -I /python /g' /kaggle/working/lifemate/scripts/*.sh
echo "--- đã khôi phục?"; ls /kaggle/working/checkpoints /kaggle/working/checkpoints/q4b_lora | head

In [ ]:
%%bash
# P-2: BƯỚC 1 của HUONG_DAN: gỡ torchao, in phiên bản torch/CUDA trong phiên CÓ GPU. GỬI NGUYÊN DÒNG ĐẦU cho Phương.
pip uninstall -y torchao 2>&1 | tail -1
python - <<'EOF' | tee /kaggle/working/torch_cuda.txt
import sys
import torch
print(torch.__version__, torch.version.cuda, sys.version)
import peft, transformers
print("peft", peft.__version__, "| transformers", transformers.__version__, "| gpus", torch.cuda.device_count(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "KHÔNG CÓ GPU")
if not torch.cuda.is_available():
    raise SystemExit("Phiên này KHÔNG có GPU: bật Accelerator rồi chạy lại (nếu không, dòng torch ở trên là bản CPU và vô nghĩa)")
EOF

In [ ]:
%%bash
# P-3: ghi sẵn hai đoạn mã phụ (chỉ in số đếm tổng hợp, không in nội dung dữ liệu)
mkdir -p /kaggle/working/helpers
cat > /kaggle/working/helpers/fr.py <<'PYEOF'
"""Chỉ in SỐ ĐẾM tổng hợp, không in nội dung câu nói hay dự đoán.
usage: python fr.py <ids_file> <submission.jsonl> [<submission.jsonl> ...]"""
import json
import sys
from pathlib import Path

ids = Path(sys.argv[1]).read_text().split()
gold = {}
with open("data/raw/train+public_test/train/train.jsonl", encoding="utf-8") as f:
    for line in f:
        d = json.loads(line)
        gold[d["id"]] = d["label"]["tool_calls"]


def load(path):
    with open(path, encoding="utf-8") as f:
        return {json.loads(line)["id"]: json.loads(line)["tool_calls"] for line in f}


def is_ask(calls):
    return any(c["name"] == "sys.ask_missing_info" for c in calls)


n_gold_call = sum(1 for i in ids if gold[i])
n_oos = sum(1 for i in ids if not gold[i])
print(f"{len(ids)} mẫu: {n_gold_call} có call, {n_oos} ngoài phạm vi (oos)")
print(f"{'file':44} {'từ chối nhầm':>14} {'oos đúng':>10} {'dự đoán ask':>12} {'call TB':>8}")
for p in sys.argv[2:]:
    if not Path(p).exists():
        print(f"{p:44} (không có file)")
        continue
    r = load(p)
    refused = sum(1 for i in ids if gold[i] and not r.get(i))
    oos_ok = sum(1 for i in ids if not gold[i] and not r.get(i))
    asks = sum(1 for i in ids if r.get(i) and is_ask(r[i]))
    avg = sum(len(r.get(i, [])) for i in ids) / len(ids)
    print(f"{Path(p).name:44} {refused:>7}/{n_gold_call:<6} {oos_ok:>5}/{n_oos:<4} {asks:>12} {avg:>8.2f}")
PYEOF
cat > /kaggle/working/helpers/pack_zip.py <<'PYEOF'
"""Đóng gói file kết quả nhỏ thành một zip kèm MANIFEST sha256 (không đóng gói trọng số).
usage: python pack_zip.py <out.zip> <file> [<file> ...]   (file không tồn tại thì bỏ qua và báo)"""
import hashlib
import sys
import zipfile
from pathlib import Path


def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while block := f.read(1 << 20):
            h.update(block)
    return h.hexdigest()


out = Path(sys.argv[1])
files = []
for name in sys.argv[2:]:
    p = Path(name)
    if p.is_file():
        files.append(p)
    else:
        print("bỏ qua (không có):", name)
files = sorted(set(files), key=str)
manifest = "".join(f"{sha256(p)}  {p.as_posix()}  ({p.stat().st_size} bytes)\n" for p in files)
with zipfile.ZipFile(out, "w", zipfile.ZIP_DEFLATED) as z:
    for p in files:
        z.write(p, p.as_posix().lstrip("/"))
    z.writestr("MANIFEST_sha256.txt", manifest)
print(f"{out}: {out.stat().st_size / 1e6:.2f} MB, {len(files)} file")
PYEOF
ls -la /kaggle/working/helpers

In [ ]:
%%bash
# A4-1: kiểm tra đủ đồ. Cần code MỚI (callgen), bản ghi ASR gốc của train, dev_ids. THIẾU thì dừng
set -uo pipefail
source /kaggle/working/lifemate/kaggle_env.sh
MISSING=0
need() { if [ -s "$1" ]; then echo "ok     $1"; else echo "THIẾU  $1"; MISSING=1; fi; }
need src/lifemate/callgen.py
need src/lifemate/leakcheck.py
need scripts/preflight_train.py
need scripts/compare_runs.py
need data/processed/asr_phowl_train.jsonl
need data/processed/dev_ids.txt
if [ "$MISSING" = 1 ]; then echo "-> Thiếu file: dùng lifemate-code BẢN MỚI (zip mới) và gắn output notebook Hello_done."; exit 1; fi
echo; echo "== q4b_lora KHÔNG được ghi đè (mọi lần chạy dùng RUN mới). sha256 hiện tại:"
sha256sum checkpoints/q4b_lora/adapter_model.safetensors 2>/dev/null || echo "(không có, không sao với A4)"
echo; df -h /kaggle/working | tail -1
mkdir -p outputs/logs

In [ ]:
%%bash
# A4-2: dựng dữ liệu "giấu 6 app" (ghi vào data/processed/holdout/, không đụng file chính). Kỳ vọng: 1180 train, 174 eval
set -euo pipefail
source /kaggle/working/lifemate/kaggle_env.sh
python -X utf8 scripts/build_sft.py --asr-tag phowl --hold-apps flights,bike_rental,package_tracking,shopping_list,podcast_radio,loyalty --hist-max 300 2>&1 | cut -c1-300
H=data/processed/holdout
wc -l $H/sft_train.jsonl $H/sft_train_asr.jsonl $H/sft_train_hist.jsonl $H/eval_ids.txt
if [ "$(wc -l < $H/sft_train.jsonl)" != 1180 ] || [ "$(wc -l < $H/eval_ids.txt)" != 174 ]; then
  echo "SỐ MẪU KHÁC KỲ VỌNG (1180 train / 174 eval): dừng, gửi output này cho Phương"; exit 1
fi
python -X utf8 scripts/preflight_train.py --data $H/sft_train.jsonl $H/sft_train_asr.jsonl $H/sft_train_hist.jsonl \
  --dev-ids $H/dev_ids.txt --hold-apps flights,bike_rental,package_tracking,shopping_list,podcast_radio,loyalty --max-len 5120 --tokenizer $QWEN

In [ ]:
%%bash
# A4-3: TRAIN q4b_holdout (mốc: chưa thấy app nào trong 6 app). Đúng lệnh của TASKS.md A4. Khoảng 2 giờ
set -euo pipefail
source /kaggle/working/lifemate/kaggle_env.sh
H=data/processed/holdout
python -X utf8 scripts/train_sft.py --model $QWEN --data $H/sft_train.jsonl $H/sft_train_asr.jsonl \
  --out checkpoints/q4b_holdout --epochs 3 --max-len 5120 --global-batch 16 --save-limit 2 2>&1 | tee outputs/logs/a4_q4b_holdout_train.log | tail -5
ls checkpoints/q4b_holdout

In [ ]:
%%bash
# A4-4: đo q4b_holdout trên các mẫu dùng app bị giấu (transcript sạch)
set -euo pipefail
source /kaggle/working/lifemate/kaggle_env.sh
H=data/processed/holdout
bash scripts/infer_multi_gpu.sh train outputs/unseen_q4b_holdout.jsonl --ids $H/eval_ids.txt --backend hf --model $QWEN --adapter checkpoints/q4b_holdout --text clean
python -X utf8 scripts/eval.py --sub outputs/unseen_q4b_holdout.jsonl --ids $H/eval_ids.txt | tee outputs/logs/a4_q4b_holdout_eval.txt
tail -qn 1 outputs/unseen_q4b_holdout.part*.log

In [ ]:
%%bash
# A4-5: TRAIN q4b_holdout_hist (như trên + 300 mẫu lịch sử). Khoảng 2 giờ.
# Thiếu giờ GPU thì đặt DO_HIST=0 (HUONG_DAN: khi đó chỉ chạy q4b_holdout)
DO_HIST=1
[ "$DO_HIST" = 1 ] || { echo "bỏ qua q4b_holdout_hist"; exit 0; }
set -euo pipefail
source /kaggle/working/lifemate/kaggle_env.sh
H=data/processed/holdout
python -X utf8 scripts/train_sft.py --model $QWEN --data $H/sft_train.jsonl $H/sft_train_asr.jsonl $H/sft_train_hist.jsonl \
  --out checkpoints/q4b_holdout_hist --epochs 3 --max-len 5120 --global-batch 16 --save-limit 2 2>&1 | tee outputs/logs/a4_q4b_holdout_hist_train.log | tail -5
ls checkpoints/q4b_holdout_hist

In [ ]:
%%bash
# A4-6: đo q4b_holdout_hist
DO_HIST=1
[ "$DO_HIST" = 1 ] || { echo "bỏ qua"; exit 0; }
set -euo pipefail
source /kaggle/working/lifemate/kaggle_env.sh
H=data/processed/holdout
bash scripts/infer_multi_gpu.sh train outputs/unseen_q4b_holdout_hist.jsonl --ids $H/eval_ids.txt --backend hf --model $QWEN --adapter checkpoints/q4b_holdout_hist --text clean
python -X utf8 scripts/eval.py --sub outputs/unseen_q4b_holdout_hist.jsonl --ids $H/eval_ids.txt | tee outputs/logs/a4_q4b_holdout_hist_eval.txt
tail -qn 1 outputs/unseen_q4b_holdout_hist.part*.log

In [ ]:
%%bash
# A4-7: so từng cặp mẫu (CHỈ so hai cấu hình A4 với nhau, không so với 56,98) + đếm từ chối nhầm trên app lạ
set -uo pipefail
source /kaggle/working/lifemate/kaggle_env.sh
H=data/processed/holdout
{
echo "################ q4b_holdout -> q4b_holdout_hist (từng cặp mẫu)"
if [ -s outputs/unseen_q4b_holdout_hist.jsonl ]; then
  python -X utf8 scripts/compare_runs.py --a outputs/unseen_q4b_holdout.jsonl --b outputs/unseen_q4b_holdout_hist.jsonl --ids $H/eval_ids.txt --names q4b_holdout,q4b_holdout_hist
else echo "(không có q4b_holdout_hist)"; fi
echo; echo "################ Từ chối nhầm trên app lạ"
python -X utf8 /kaggle/working/helpers/fr.py $H/eval_ids.txt outputs/unseen_q4b_holdout.jsonl outputs/unseen_q4b_holdout_hist.jsonl
} 2>&1 | tee outputs/logs/a4_compare.txt

In [ ]:
%%bash
# A4-8: gói kết quả nhỏ để gửi Phương: a4_results.zip (điểm, file unseen_*.jsonl, log, torch_cuda.txt, sha256 adapter)
set -uo pipefail
source /kaggle/working/lifemate/kaggle_env.sh
for d in q4b_holdout q4b_holdout_hist; do [ -f checkpoints/$d/adapter_model.safetensors ] && sha256sum checkpoints/$d/adapter_model.safetensors; done | tee outputs/logs/a4_adapter_sha256.txt
python -X utf8 /kaggle/working/helpers/pack_zip.py /kaggle/working/a4_results.zip outputs/unseen_*.jsonl outputs/logs/a4_*.txt outputs/logs/a4_*.log /kaggle/working/torch_cuda.txt
du -sh /kaggle/working/checkpoints /kaggle/working/outputs; df -h /kaggle/working | tail -1

In [ ]:
%%bash
# A4-9 (CUỐI CÙNG): lưu trạng thái rồi Save Version. Cell này xóa thư mục làm việc lifemate/.
bash /kaggle/working/lifemate/scripts/kaggle_save_state.sh
du -sh /kaggle/working/state /kaggle/working/checkpoints /kaggle/working/outputs